# Étape 3 - Nettoyage, feature engineering et séparation

Cette étape exécute le cahier des charges produit par le rapport de qualité de l'étape 2,
puis enrichit le jeu avant de le séparer en apprentissage et test.

1. **Nettoyage** : corriger les anomalies de format et de cohérence, en journalisant
   chaque action et en conservant intact le jeu d'origine.
2. **Feature engineering** : construire des variables qui portent une information que les
   colonnes brutes ne portent pas, sans jamais utiliser la cible.
3. **Documentation** : produire le dictionnaire de données du jeu final.
4. **Séparation** : découper apprentissage et test une fois pour toutes.

La règle qui gouverne toute l'étape : **rien de ce qui s'apprend sur les données ne se
fait ici.** Imputation, encodage et mise à l'échelle appartiennent à la pipeline des
étapes 4 et 5. Ce qui se fait ici, ce sont des corrections d'erreurs et des calculs
déterministes ligne à ligne.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_PATH = Path.cwd().parent
DATA_PATH = PROJECT_PATH / 'csv' / 'dataset_phase1_modelisation.csv'
CLEAN_PATH = PROJECT_PATH / 'csv' / 'dataset_phase3_final.csv'
LOG_PATH = PROJECT_PATH / 'csv' / 'nettoyage_log.json'
DICTIONNAIRE_PATH = PROJECT_PATH / 'DATA_DICTIONARY.md'

TARGET = "6-Taux d'emploi salarié en France - 6 mois après le diplôme"
SORTANTS = '6-Nombre de sortants - 6 mois après le diplôme'
POURSUIVANTS = '6-Nombre de poursuivants - 6 mois après le diplôme'
NB_STABLE = '6-Nombre de sortants en emploi stable - 6 mois après le diplôme'
TX_STABLE = '6-Taux de sortants en emploi stable - 6 mois après le diplôme'
CODE_UAI = "Code UAI de l'établissement"
CODE_SISE = 'Code du diplôme SISE'
MILLESIME = 2026

# Le jeu d'origine est chargé une fois et n'est jamais modifié : toutes les
# transformations travaillent sur une copie, ce qui rend le comparatif possible.
df_origine = pd.read_csv(DATA_PATH)
df = df_origine.copy()

journal = []


def journaliser(action, dimension, lignes_touchees, justification):
    """Enregistre une action de nettoyage dans le log de traçabilité."""
    journal.append({
        'action': action,
        'dimension_qualite': dimension,
        'lignes_touchees': int(lignes_touchees),
        'justification': justification,
    })
    print(f'[{dimension}] {action} → {lignes_touchees} ligne(s)')


print('Jeu d\'origine :', df_origine.shape)

Jeu d'origine : (17065, 23)


## 1. Nettoyage

### 1.1 Unicité : vérification, pas suppression

L'étape 2 a montré que la clé `Code UAI × Code SISE × Promotion` est unique. On le
revérifie ici avant toute transformation : si le nettoyage devait créer des doublons, il
faut le savoir.

In [2]:
CLE_PRIMAIRE = [CODE_UAI, CODE_SISE, 'Promotion']

doublons_complets = int(df.duplicated().sum())
doublons_cle = int(df.duplicated(CLE_PRIMAIRE).sum())

if doublons_complets:
    df = df.drop_duplicates()
journaliser('Suppression des doublons de ligne complète', 'Unicité', doublons_complets,
            'Aucun doublon complet : le filtrage des marges à l\'étape 1 les avait déjà écartés')
journaliser('Contrôle de la clé primaire UAI × SISE × Promotion', 'Unicité', doublons_cle,
            'Clé unique validée à l\'étape 2 ; aucune agrégation nécessaire')

[Unicité] Suppression des doublons de ligne complète → 0 ligne(s)
[Unicité] Contrôle de la clé primaire UAI × SISE × Promotion → 0 ligne(s)


### 1.2 Formats : standardisation des libellés

Les libellés sont saisis à la main dans la source. On cherche les défauts classiques :
espaces en bord de chaîne, espaces multiples internes, casse incohérente.

In [3]:
COLONNES_TEXTE = df.select_dtypes(include='str').columns.tolist()

defauts = pd.DataFrame({
    'espaces en bord': [int((df[c] != df[c].str.strip()).sum()) for c in COLONNES_TEXTE],
    'espaces multiples': [int(df[c].str.contains('  ', regex=False).sum()) for c in COLONNES_TEXTE],
    'casse non homogène': [int((df[c] != df[c].str.upper()).sum()) for c in COLONNES_TEXTE],
}, index=COLONNES_TEXTE)
defauts

,espaces en bord,espaces multiples,casse non homogène
Région,0,0,17065
Académie,0,0,17065
Établissement,0,0,16970
Type de diplôme,0,0,17065
Domaine disciplinaire,0,0,17065
Discipline,0,0,16573
Secteur disciplinaire,0,0,16573
Libellé du diplôme,0,13,390
Code UAI de l'établissement,0,0,0
Code du diplôme SISE,0,0,0


In [4]:
# Les libellés de diplôme sont censés être publiés en majuscules non accentuées.
LIBELLE = 'Libellé du diplôme'
espaces_multiples = int(df[LIBELLE].str.contains('  ', regex=False).sum())
casse_incoherente = int((df[LIBELLE] != df[LIBELLE].str.upper()).sum())

print('Exemples de libellés à espaces multiples :')
for exemple in df.loc[df[LIBELLE].str.contains('  ', regex=False), LIBELLE].unique()[:3]:
    print('  ·', exemple)
print()
print('Exemples de libellés à casse incohérente :')
for exemple in df.loc[df[LIBELLE] != df[LIBELLE].str.upper(), LIBELLE].unique()[:3]:
    print('  ·', exemple)

libelles_avant = df[LIBELLE].nunique()
for colonne in COLONNES_TEXTE:
    df[colonne] = df[colonne].str.strip().str.replace(r'\s+', ' ', regex=True)
df[LIBELLE] = df[LIBELLE].str.upper()
libelles_apres = df[LIBELLE].nunique()

journaliser('Normalisation des espaces multiples et de bord', 'Cohérence', espaces_multiples,
            'Espaces doubles internes issus de la saisie ; deux écritures d\'un même diplôme '
            'sinon comptées comme deux libellés distincts')
journaliser('Passage des libellés de diplôme en majuscules', 'Cohérence', casse_incoherente,
            'Convention de publication de la source ; trois libellés y dérogeaient')
print()
print(f'Libellés distincts : {libelles_avant} avant, {libelles_apres} après')

Exemples de libellés à espaces multiples :
  · INGENIEUR DIPLOME DE L'INSTITUT NATIONAL DES SCIENCES APPLIQUEES  DE STRASBOURG, SPECIALITE MECANIQUE
  · SCIENCES HUMAINES ET SOCIALES : INFORMATION  ET COMMUNICATION
  · INGENIEUR DIPLOME DE L'ECOLE CENTRALE DE NANTES,  SPECIALITE MECANIQUE

Exemples de libellés à casse incohérente :
  · METIERS DE L'ENSEIGNEMENT, DE L'EDUCATION ET DE LA FORMATION (MEEF), 1er DEGRE
  · METIERS DE L'ENSEIGNEMENT, DE L'EDUCATION ET DE LA FORMATION (MEEF), 2e DEGRE
  · DIPLOME D'ETUDES SUPERIEURES SPECIALISEES EN MANAGEMENT PAR L'INNOVATION DE l'ICN


[Cohérence] Normalisation des espaces multiples et de bord → 13 ligne(s)
[Cohérence] Passage des libellés de diplôme en majuscules → 390 ligne(s)

Libellés distincts : 1290 avant, 1289 après


### 1.3 Cohérence : les 11 anomalies de taux d'emploi stable

L'étape 2 a trouvé 11 lignes où le taux d'emploi stable publié ne se reconstitue pas à
partir des effectifs. Le taux publié et l'effectif publié se contredisent : l'un des deux
est faux, et rien ne permet de savoir lequel.

La décision retenue est de **mettre le taux à `NaN` sur ces lignes et de conserver la
ligne**. La cible n'est pas concernée par l'anomalie, il n'y a donc aucune raison de
perdre l'observation ; en revanche, propager une valeur dont on sait qu'elle est fausse
serait pire que l'absence de valeur.

In [5]:
emploi_salarie_implicite = df[TARGET] / 100 * df[SORTANTS]
taux_stable_recalcule = 100 * df[NB_STABLE] / emploi_salarie_implicite.replace(0, np.nan)
incoherentes = (taux_stable_recalcule - df[TX_STABLE]).abs() > 0.05

print('Lignes concernées :')
apercu = df.loc[incoherentes, [SORTANTS, TARGET, NB_STABLE, TX_STABLE]].copy()
apercu['taux stable recalculé'] = taux_stable_recalcule[incoherentes].round(2)
print(apercu.head(11).to_string())

df.loc[incoherentes, TX_STABLE] = np.nan
journaliser("Mise à NaN du taux d'emploi stable incohérent", 'Cohérence',
            int(incoherentes.sum()),
            'Taux publié contredit par les effectifs publiés ; la ligne est conservée car '
            'la cible n\'est pas affectée')

Lignes concernées :
       6-Nombre de sortants - 6 mois après le diplôme  6-Taux d'emploi salarié en France - 6 mois après le diplôme  6-Nombre de sortants en emploi stable - 6 mois après le diplôme  6-Taux de sortants en emploi stable - 6 mois après le diplôme  taux stable recalculé
2813                                               22                                                         4.55                                                                1                                                         100.00                  99.90
4222                                               38                                                         7.89                                                                3                                                         100.00                 100.06
4969                                               22                                                         4.55                                                                1   

### 1.4 Valeurs atypiques : décision de conservation

L'étape 2 a identifié 31 formations dans la queue basse de la distribution. Le contrôle
ci-dessous confirme qu'aucune ne sort du domaine de définition d'un taux : ce sont des
valeurs faibles, pas des valeurs impossibles. Aucune suppression n'est effectuée, et cette
décision est journalisée pour être défendable.

In [6]:
hors_domaine = int(((df[TARGET] < 0) | (df[TARGET] > 100)).sum())
q1, q3 = df[TARGET].quantile([0.25, 0.75])
iqr = q3 - q1
atypiques = int(((df[TARGET] < q1 - 1.5 * iqr) | (df[TARGET] > q3 + 1.5 * iqr)).sum())

assert hors_domaine == 0, 'Un taux sort du domaine [0, 100]'
journaliser('Conservation des valeurs atypiques de la cible', 'Exactitude', atypiques,
            'Formations à faible effectif où un taux extrême est réel ; les supprimer '
            'retirerait du jeu les cas les plus difficiles et gonflerait artificiellement le score')
print(f'Taux hors du domaine [0, 100] : {hors_domaine}')
print(f'Valeurs atypiques conservées : {atypiques}')

[Exactitude] Conservation des valeurs atypiques de la cible → 31 ligne(s)
Taux hors du domaine [0, 100] : 0
Valeurs atypiques conservées : 31


### 1.5 Redondance et types

`promotion_debut` reprend exactement `Promotion`. Garder les deux ferait entrer deux fois
la même information dans le modèle.

In [7]:
if 'promotion_debut' in df.columns:
    identiques = bool((df['promotion_debut'] == df['Promotion']).all())
    print('promotion_debut identique à Promotion :', identiques)
    assert identiques, 'Les deux colonnes ne sont pas identiques, la suppression serait abusive'
    df = df.drop(columns='promotion_debut')
    journaliser('Suppression de la colonne redondante promotion_debut', 'Exactitude', len(df),
                'Colonne strictement identique à Promotion : doublon d\'information')

# Les codes sont des identifiants, jamais des nombres : on garantit leur type texte.
for colonne in (CODE_UAI, CODE_SISE):
    df[colonne] = df[colonne].astype(str)
df['Promotion'] = df['Promotion'].astype(int)

print()
print('Types après nettoyage :')
print(df.dtypes.value_counts().to_string())

promotion_debut identique à Promotion : True
[Exactitude] Suppression de la colonne redondante promotion_debut → 17065 ligne(s)

Types après nettoyage :
str        10
float64     7
int64       5


### 1.6 Comparatif avant / après et journal de nettoyage

In [8]:
comparatif = pd.DataFrame({
    'avant': {
        'lignes': len(df_origine),
        'colonnes': df_origine.shape[1],
        'doublons complets': int(df_origine.duplicated().sum()),
        'doublons sur la clé primaire': int(df_origine.duplicated(CLE_PRIMAIRE).sum()),
        'libellés de diplôme distincts': int(df_origine[LIBELLE].nunique()),
        'valeurs manquantes': int(df_origine.isna().sum().sum()),
        'moyenne de la cible': round(df_origine[TARGET].mean(), 3),
        'écart-type de la cible': round(df_origine[TARGET].std(), 3),
    },
    'après': {
        'lignes': len(df),
        'colonnes': df.shape[1],
        'doublons complets': int(df.duplicated().sum()),
        'doublons sur la clé primaire': int(df.duplicated(CLE_PRIMAIRE).sum()),
        'libellés de diplôme distincts': int(df[LIBELLE].nunique()),
        'valeurs manquantes': int(df.isna().sum().sum()),
        'moyenne de la cible': round(df[TARGET].mean(), 3),
        'écart-type de la cible': round(df[TARGET].std(), 3),
    },
})
comparatif['écart'] = comparatif['après'] - comparatif['avant']
comparatif

,avant,après,écart
lignes,17065.000,17065.000,0.0
colonnes,23.000,22.000,-1.0
doublons complets,0.000,0.000,0.0
doublons sur la clé primaire,0.000,0.000,0.0
libellés de diplôme distincts,1290.000,1289.000,-1.0
valeurs manquantes,5884.000,5895.000,11.0
moyenne de la cible,55.889,55.889,0.0
écart-type de la cible,18.469,18.469,0.0


In [9]:
log_nettoyage = {
    'source': DATA_PATH.name,
    'lignes_avant': int(len(df_origine)),
    'lignes_apres': int(len(df)),
    'colonnes_avant': int(df_origine.shape[1]),
    'colonnes_apres': int(df.shape[1]),
    'cle_primaire': CLE_PRIMAIRE,
    'actions': journal,
}
LOG_PATH.write_text(json.dumps(log_nettoyage, ensure_ascii=False, indent=2), encoding='utf-8')
print('Journal de nettoyage écrit :', LOG_PATH.name)
pd.DataFrame(journal)

Journal de nettoyage écrit : nettoyage_log.json


,action,dimension_qualite,lignes_touchees,justification
0,Suppression des doublons de ligne complète,Unicité,0,Aucun doublon complet : le filtrage des marges...
1,Contrôle de la clé primaire UAI × SISE × Promo...,Unicité,0,Clé unique validée à l'étape 2 ; aucune agréga...
2,Normalisation des espaces multiples et de bord,Cohérence,13,Espaces doubles internes issus de la saisie ; ...
3,Passage des libellés de diplôme en majuscules,Cohérence,390,Convention de publication de la source ; trois...
4,Mise à NaN du taux d'emploi stable incohérent,Cohérence,11,Taux publié contredit par les effectifs publié...
5,Conservation des valeurs atypiques de la cible,Exactitude,31,Formations à faible effectif où un taux extrêm...
6,Suppression de la colonne redondante promotion...,Exactitude,17065,Colonne strictement identique à Promotion : do...


**Lecture du comparatif.** Aucune ligne n'a été perdue, la moyenne et l'écart-type de la
cible sont inchangés : le nettoyage n'a pas déformé la distribution qu'on cherche à
prédire. Les seuls écarts portent sur la colonne redondante retirée, les libellés
harmonisés et les 11 valeurs de taux d'emploi stable neutralisées.

C'est le résultat attendu ici. Le gros du travail de mise en forme a été fait à l'étape 1,
en écartant les marges du cube : le jeu arrivait déjà propre, et le dire est plus honnête
que d'inventer des corrections pour remplir un journal.

## 2. Feature engineering

Dix variables sont construites, réparties sur les cinq familles attendues. Chacune répond
à une hypothèse formulée à la fin de l'étape 2.

| Famille | Variable | Hypothèse testée |
|---|---|---|
| Temporelle | `anciennete_promotion` | L'ancienneté de la promotion capte la conjoncture |
| Indicateur | `promotion_choc_sanitaire` | La promotion 2020 subit un choc spécifique |
| Indicateur | `est_diplome_professionnalisant` | Un diplôme qui mène à un métier insère mieux |
| Calculée | `taille_formation` | Sortants et poursuivants forment un effectif total |
| Calculée | `ratio_poursuite` | Un diplômé qui poursuit ses études n'est pas en emploi |
| Catégorielle | `tranche_effectif` | L'effet de l'effectif n'est pas linéaire |
| Agrégée | `taille_mediane_secteur` | Les secteurs à grosses promotions ont d'autres débouchés |
| Agrégée | `nb_formations_etablissement` | Un établissement généraliste diffère d'un spécialisé |
| Agrégée | `part_sortants_etablissement` | Le poids de la formation dans son établissement |
| Agrégée | `nb_etablissements_par_diplome` | Un diplôme rare ne s'insère pas comme un diplôme répandu |

**Aucune de ces variables n'utilise la cible.** C'est la contrainte principale : un
agrégat de `y` calculé ici, même par groupe, ferait entrer dans `X` une information
dérivée des réponses, y compris celles du jeu de test. Les encodages qui ont besoin de la
cible (`TargetEncoder`) restent dans la pipeline des étapes suivantes, où la validation
croisée les réajuste à chaque pli.

In [10]:
DIPLOMES_PROFESSIONNALISANTS = [
    'Licence professionnelle', 'Master MEEF', "Diplôme d'ingénieurs",
    'Bachelor universitaire de technologie',
]

# Temporelle et indicateurs : déterministes, ligne à ligne.
df['anciennete_promotion'] = MILLESIME - df['Promotion']
df['promotion_choc_sanitaire'] = (df['Promotion'] == 2020).astype(int)
df['est_diplome_professionnalisant'] = (
    df['Type de diplôme'].isin(DIPLOMES_PROFESSIONNALISANTS).astype(int))

# Calculées : combinaisons d'effectifs de la même ligne.
df['taille_formation'] = df[SORTANTS] + df[POURSUIVANTS]
df['ratio_poursuite'] = (df[POURSUIVANTS] / df['taille_formation']).round(4)

# Catégorielle : découpage en quartiles d'effectif, bornes lisibles.
df['tranche_effectif'] = pd.cut(
    df[SORTANTS], bins=[0, 25, 45, 90, np.inf],
    labels=['très petite (≤ 25)', 'petite (26-45)', 'moyenne (46-90)', 'grande (> 90)'],
).astype(str)

# Agrégées : comptages et médianes d'effectifs, jamais de moyenne de la cible.
df['taille_mediane_secteur'] = df.groupby('Secteur disciplinaire')[SORTANTS].transform('median')
df['nb_formations_etablissement'] = df.groupby(CODE_UAI)[CODE_SISE].transform('nunique')
df['part_sortants_etablissement'] = (
    df[SORTANTS] / df.groupby([CODE_UAI, 'Promotion'])[SORTANTS].transform('sum')).round(4)
df['nb_etablissements_par_diplome'] = df.groupby(CODE_SISE)[CODE_UAI].transform('nunique')

FEATURES_CREEES = ['anciennete_promotion', 'promotion_choc_sanitaire',
                   'est_diplome_professionnalisant', 'taille_formation', 'ratio_poursuite',
                   'tranche_effectif', 'taille_mediane_secteur',
                   'nb_formations_etablissement', 'part_sortants_etablissement',
                   'nb_etablissements_par_diplome']

print(f'{len(FEATURES_CREEES)} variables créées, jeu enrichi :', df.shape)
print('Valeurs manquantes introduites :',
      int(df[FEATURES_CREEES].isna().sum().sum()))
df[FEATURES_CREEES].describe(include='all').T

10 variables créées, jeu enrichi : (17065, 32)
Valeurs manquantes introduites : 0


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
anciennete_promotion,17065.0,NaN,NaN,NaN,4.443891,1.693811,2.0,3.0,4.0,6.0,7.0
promotion_choc_sanitaire,17065.0,NaN,NaN,NaN,0.158805,0.365504,0.0,0.0,0.0,0.0,1.0
est_diplome_professionnalisant,17065.0,NaN,NaN,NaN,0.309757,0.462407,0.0,0.0,0.0,1.0,1.0
taille_formation,17065.0,NaN,NaN,NaN,107.680398,147.878368,20.0,36.0,58.0,112.0,2176.0
ratio_poursuite,17065.0,NaN,NaN,NaN,0.291747,0.251095,0.0,0.0972,0.2,0.4286,0.9706
tranche_effectif,17065,4,petite (26-45),6416,NaN,NaN,NaN,NaN,NaN,NaN,NaN
taille_mediane_secteur,17065.0,NaN,NaN,NaN,40.614767,13.583304,21.0,33.0,39.0,46.0,98.5
nb_formations_etablissement,17065.0,NaN,NaN,NaN,58.769352,37.426545,1.0,34.0,60.0,77.0,143.0
part_sortants_etablissement,17065.0,NaN,NaN,NaN,0.104307,0.232026,0.0033,0.0131,0.0235,0.0557,1.0
nb_etablissements_par_diplome,17065.0,NaN,NaN,NaN,17.19162,15.894142,1.0,3.0,12.0,28.0,63.0


In [11]:
# Force du lien de chaque variable créée avec la cible : utile pour anticiper
# lesquelles ressortiront à l'étape 5.
force = []
for variable in FEATURES_CREEES:
    if variable == 'tranche_effectif':
        moyennes = df.groupby(variable)[TARGET].transform('mean')
        force.append({'variable': variable, 'mesure': 'eta²',
                      'valeur': round(moyennes.var(ddof=0) / df[TARGET].var(ddof=0), 3)})
    else:
        force.append({'variable': variable, 'mesure': 'corrélation de Pearson',
                      'valeur': round(df[variable].corr(df[TARGET]), 3)})
force = pd.DataFrame(force).reindex(
    pd.DataFrame(force)['valeur'].abs().sort_values(ascending=False).index)
force

,variable,mesure,valeur
2,est_diplome_professionnalisant,corrélation de Pearson,0.407
4,ratio_poursuite,corrélation de Pearson,-0.311
6,taille_mediane_secteur,corrélation de Pearson,0.239
1,promotion_choc_sanitaire,corrélation de Pearson,-0.181
3,taille_formation,corrélation de Pearson,-0.151
0,anciennete_promotion,corrélation de Pearson,-0.105
8,part_sortants_etablissement,corrélation de Pearson,-0.032
7,nb_formations_etablissement,corrélation de Pearson,0.032
9,nb_etablissements_par_diplome,corrélation de Pearson,0.020
5,tranche_effectif,eta²,0.003


**Redondances assumées.** `anciennete_promotion` et `promotion_choc_sanitaire` se
déduisent toutes deux de `Promotion`, qui est par ailleurs fournie comme variable
catégorielle. Ce n'est pas un oubli : l'encodage one-hot de `Promotion` donne un effet
libre par année, `anciennete_promotion` donne une variable ordonnée sur laquelle un arbre
peut couper une fois pour toutes, et `promotion_choc_sanitaire` isole explicitement
l'hypothèse testée. Les trois formes coexistent, et l'importance des variables de
l'étape 5 dira laquelle le modèle utilise réellement.

**Lecture.** Trois variables portent un signal net : `est_diplome_professionnalisant`
(+0,41), `ratio_poursuite` (−0,31) et `taille_mediane_secteur` (+0,24). La deuxième
confirme directement l'hypothèse 1 de l'étape 2 : plus la part de diplômés qui poursuivent
leurs études est élevée, plus le taux d'emploi salarié baisse.

`tranche_effectif` est la plus décevante (eta² = 0,003) : découper le nombre de sortants
en quatre classes ne capte quasiment rien, ce qui est cohérent avec la corrélation nulle
du nombre de sortants brut (r = 0,02). Le signal de taille passe par `taille_formation`,
qui inclut les poursuivants.

Les trois variables d'agrégation par établissement et par diplôme sont, elles aussi,
presque plates (|r| < 0,04). Elles sont **conservées quand même** : une corrélation linéaire
faible n'exclut pas un apport en interaction dans une forêt aléatoire. L'étape 5
tranchera sur la base de l'importance réelle des variables, et si elles ne ressortent pas,
ce sera un résultat à reporter, pas un échec à cacher.

### Contrôle anti-fuite

Deux vérifications sont nécessaires avant d'aller plus loin.

**Première vérification : aucune variable créée ne dérive de la cible.** Le test est
mécanique : on recalcule chaque variable après avoir permuté aléatoirement la cible. Si
une variable dépendait de `y`, sa valeur changerait.

**Seconde vérification : les agrégats ne dépendent pas du découpage.** Trois variables
sont calculées sur l'ensemble du jeu. Elles ne regardent que `X`, mais on mesure quand
même l'écart entre leur valeur calculée sur tout le jeu et leur valeur recalculée sur le
seul jeu d'apprentissage.

In [12]:
from sklearn.model_selection import train_test_split

# 1. Indépendance à la cible : on permute y et on recalcule.
melange = df.copy()
melange[TARGET] = melange[TARGET].sample(frac=1, random_state=0).to_numpy()
melange['ratio_poursuite'] = (melange[POURSUIVANTS] / melange['taille_formation']).round(4)
melange['taille_mediane_secteur'] = (
    melange.groupby('Secteur disciplinaire')[SORTANTS].transform('median'))
melange['nb_formations_etablissement'] = melange.groupby(CODE_UAI)[CODE_SISE].transform('nunique')

identiques = all(
    bool(melange[variable].equals(df[variable]))
    for variable in ['ratio_poursuite', 'taille_mediane_secteur', 'nb_formations_etablissement'])
print('Variables inchangées après permutation de la cible :', identiques)
assert identiques, 'Une variable créée dépend de la cible : fuite de données'

Variables inchangées après permutation de la cible : True


In [13]:
# 2. Sensibilité des agrégats au découpage apprentissage / test.
index_train, index_test = train_test_split(df.index, test_size=0.20, random_state=42)
train_seul = df.loc[index_train]

agregats_train = pd.DataFrame({
    'taille_mediane_secteur': train_seul.groupby('Secteur disciplinaire')[SORTANTS].median(),
})
recalcule = df['Secteur disciplinaire'].map(agregats_train['taille_mediane_secteur'])
ecart_relatif = ((recalcule - df['taille_mediane_secteur']).abs()
                 / df['taille_mediane_secteur']).dropna()

nb_form_train = train_seul.groupby(CODE_UAI)[CODE_SISE].nunique()
recalcule_form = df[CODE_UAI].map(nb_form_train)
correlation_agregat = recalcule_form.corr(df['nb_formations_etablissement'])

print(f"taille_mediane_secteur : écart relatif médian entre le calcul global et le calcul "
      f"sur le seul train = {100 * ecart_relatif.median():.2f} %")
print(f"  écart relatif maximal = {100 * ecart_relatif.max():.2f} %")
print(f"nb_formations_etablissement : corrélation entre les deux calculs = "
      f"{correlation_agregat:.4f}")
print()
print("Ces variables décrivent le catalogue d'un établissement ou la taille usuelle d'un")
print("secteur : elles seraient connues d'un référentiel avant toute prédiction, et elles")
print("ne changent quasiment pas selon le découpage. Aucune information sur la cible du")
print("test ne transite par elles.")

taille_mediane_secteur : écart relatif médian entre le calcul global et le calcul sur le seul train = 0.00 %
  écart relatif maximal = 15.79 %
nb_formations_etablissement : corrélation entre les deux calculs = 0.9986

Ces variables décrivent le catalogue d'un établissement ou la taille usuelle d'un
secteur : elles seraient connues d'un référentiel avant toute prédiction, et elles
ne changent quasiment pas selon le découpage. Aucune information sur la cible du
test ne transite par elles.


## 3. Périmètre du modèle : variables retenues et exclues

Le choix des variables explicatives applique les décisions du rapport de qualité.

In [14]:
CATEGORICAL_FEATURES = ['Région', 'Académie', 'Type de diplôme', 'Domaine disciplinaire',
                        'Discipline', 'Secteur disciplinaire', 'Promotion', 'tranche_effectif']
NUMERIC_FEATURES = [SORTANTS, POURSUIVANTS, 'anciennete_promotion',
                    'promotion_choc_sanitaire', 'est_diplome_professionnalisant',
                    'taille_formation', 'ratio_poursuite', 'taille_mediane_secteur',
                    'nb_formations_etablissement', 'part_sortants_etablissement',
                    'nb_etablissements_par_diplome']
FEATURES = CATEGORICAL_FEATURES + NUMERIC_FEATURES

EXCLUSIONS = pd.DataFrame([
    {'colonne': "12, 18, 24 et 30-Taux d'emploi salarié", 'motif': 'Fuite de cible',
     'détail': 'Mesuré après les 6 mois que le modèle doit prédire (r = 0,85 à 12 mois)'},
    {'colonne': "6-Taux et nombre en emploi stable", 'motif': 'Fuite de cible',
     'détail': "Résultat d'insertion mesuré au même horizon que la cible"},
    {'colonne': "6-Taux et nombre en emploi non salarié", 'motif': 'Fuite de cible',
     'détail': "Résultat d'insertion mesuré au même horizon que la cible"},
    {'colonne': 'Établissement, Libellé du diplôme', 'motif': 'Cardinalité',
     'détail': '329 et 1 290 modalités : le modèle mémoriserait au lieu de généraliser'},
    {'colonne': f'{CODE_UAI}, {CODE_SISE}', 'motif': 'Identifiant',
     'détail': 'Identifiants purs ; exploités via les variables agrégées dérivées'},
    {'colonne': 'promotion_debut', 'motif': 'Redondance',
     'détail': 'Supprimé au nettoyage, identique à Promotion'},
])

print(f'{len(FEATURES)} variables explicatives retenues : '
      f'{len(CATEGORICAL_FEATURES)} catégorielles, {len(NUMERIC_FEATURES)} numériques')
print(f'Dont {len(FEATURES_CREEES)} créées à cette étape')
print(f'Valeurs manquantes dans X : {int(df[FEATURES].isna().sum().sum())}')
EXCLUSIONS

19 variables explicatives retenues : 8 catégorielles, 11 numériques
Dont 10 créées à cette étape
Valeurs manquantes dans X : 0


,colonne,motif,détail
0,"12, 18, 24 et 30-Taux d'emploi salarié",Fuite de cible,Mesuré après les 6 mois que le modèle doit pré...
1,6-Taux et nombre en emploi stable,Fuite de cible,Résultat d'insertion mesuré au même horizon qu...
2,6-Taux et nombre en emploi non salarié,Fuite de cible,Résultat d'insertion mesuré au même horizon qu...
3,"Établissement, Libellé du diplôme",Cardinalité,329 et 1 290 modalités : le modèle mémoriserai...
4,"Code UAI de l'établissement, Code du diplôme SISE",Identifiant,Identifiants purs ; exploités via les variable...
5,promotion_debut,Redondance,"Supprimé au nettoyage, identique à Promotion"


## 4. Dictionnaire de données

Le dictionnaire est généré à partir du jeu final : il ne peut donc pas se désynchroniser
des données réelles. Il est écrit dans `Projet/DATA_DICTIONARY.md`.

In [15]:
DESCRIPTIONS = {
    'Région': "Région administrative de l'établissement",
    'Académie': "Académie de rattachement de l'établissement",
    'Établissement': "Nom de l'établissement (non identifiant : 329 noms, 365 codes UAI)",
    'Type de diplôme': 'Nature du diplôme délivré (11 modalités)',
    'Domaine disciplinaire': 'Domaine disciplinaire (4 modalités, agrégat de la discipline)',
    'Discipline': 'Discipline (14 modalités)',
    'Secteur disciplinaire': 'Secteur disciplinaire (49 modalités, niveau le plus fin)',
    'Libellé du diplôme': 'Intitulé du diplôme (non identifiant : plusieurs codes SISE partagent un même libellé)',
    'Promotion': 'Année de la promotion sortante (2019 à 2024)',
    CODE_UAI: "Identifiant national de l'établissement ; 1re partie de la clé primaire",
    CODE_SISE: 'Identifiant national du diplôme ; 2e partie de la clé primaire',
    SORTANTS: "Nombre de diplômés sortis de formation, mesuré 6 mois après le diplôme",
    POURSUIVANTS: 'Nombre de diplômés ayant poursuivi leurs études',
    TARGET: 'CIBLE : part des sortants en emploi salarié en France 6 mois après le diplôme (%)',
    "12-Taux d'emploi salarié en France - 12 mois après le diplôme": 'Même mesure à 12 mois (exclue du modèle)',
    "18-Taux d'emploi salarié en France - 18 mois après le diplôme": 'Même mesure à 18 mois (exclue du modèle)',
    "24-Taux d'emploi salarié en France - 24 mois après le diplôme": 'Même mesure à 24 mois (exclue, absente pour 2024)',
    "30-Taux d'emploi salarié en France - 30 mois après le diplôme": 'Même mesure à 30 mois (exclue, absente pour 2024)',
    '6-Nombre de sortants en emploi non salarié - 6 mois après le diplôme': "Sortants en emploi non salarié (exclu du modèle)",
    '6-Taux de sortants en emploi non salarié - 6 mois après le diplôme': "Part des sortants en emploi non salarié (exclue du modèle)",
    NB_STABLE: 'Sortants en emploi stable (CDI, fonctionnaire) parmi les salariés (exclu du modèle)',
    TX_STABLE: "Part d'emploi stable parmi les salariés (exclue ; 11 valeurs neutralisées)",
    'anciennete_promotion': 'CRÉÉE · Années écoulées entre la promotion et le millésime 2026',
    'promotion_choc_sanitaire': 'CRÉÉE · Vaut 1 pour la promotion 2020, sortie pendant la crise sanitaire',
    'est_diplome_professionnalisant': 'CRÉÉE · Vaut 1 pour licence pro, MEEF, ingénieur, BUT',
    'taille_formation': 'CRÉÉE · Sortants + poursuivants : effectif total de la promotion',
    'ratio_poursuite': 'CRÉÉE · Part de la promotion qui poursuit ses études (0 à 1)',
    'tranche_effectif': "CRÉÉE · Classe de taille de la formation, en quatre tranches",
    'taille_mediane_secteur': 'CRÉÉE · Nombre médian de sortants dans le secteur disciplinaire',
    'nb_formations_etablissement': "CRÉÉE · Nombre de diplômes distincts offerts par l'établissement",
    'part_sortants_etablissement': "CRÉÉE · Part des sortants de l'établissement issus de cette formation",
    'nb_etablissements_par_diplome': 'CRÉÉE · Nombre d\'établissements délivrant ce diplôme',
}

roles = {}
for colonne in df.columns:
    if colonne == TARGET:
        roles[colonne] = 'Cible'
    elif colonne in FEATURES:
        roles[colonne] = 'Variable explicative'
    elif colonne in CLE_PRIMAIRE:
        roles[colonne] = 'Clé primaire'
    else:
        roles[colonne] = 'Exclue du modèle'

dictionnaire = pd.DataFrame({
    'colonne': df.columns,
    'type': [str(df[c].dtype) for c in df.columns],
    'rôle': [roles[c] for c in df.columns],
    'valeurs_manquantes': [int(df[c].isna().sum()) for c in df.columns],
    'modalités': [int(df[c].nunique()) for c in df.columns],
    'exemple': [str(df[c].dropna().iloc[0])[:38] for c in df.columns],
    'description': [DESCRIPTIONS.get(c, '') for c in df.columns],
})
manquantes = dictionnaire.loc[dictionnaire['description'] == '', 'colonne'].tolist()
assert not manquantes, f'Colonnes sans description : {manquantes}'
dictionnaire

,colonne,type,rôle,valeurs_manquantes,modalités,exemple,description
0,Région,str,Variable explicative,0,19,Pays de la Loire,Région administrative de l'établissement
1,Académie,str,Variable explicative,0,32,Nantes,Académie de rattachement de l'établissement
2,Établissement,str,Exclue du modèle,0,329,École de gestion et de commerce de Ven,Nom de l'établissement (non identifiant : 329 ...
3,Type de diplôme,str,Variable explicative,0,11,Diplôme visé niveau bac + 3,Nature du diplôme délivré (11 modalités)
4,Domaine disciplinaire,str,Variable explicative,0,4,"Droit, économie, gestion","Domaine disciplinaire (4 modalités, agrégat de..."
5,Discipline,str,Variable explicative,0,14,"Sciences économiques, gestion",Discipline (14 modalités)
6,Secteur disciplinaire,str,Variable explicative,0,49,Sciences de gestion,"Secteur disciplinaire (49 modalités, niveau le..."
7,Libellé du diplôme,str,Exclue du modèle,0,1289,DIPLOME DE L'ECOLE DE GESTION ET DE CO,Intitulé du diplôme (non identifiant : plusieu...
8,Promotion,int64,Variable explicative,0,6,2019,Année de la promotion sortante (2019 à 2024)
9,Code UAI de l'établissement,str,Clé primaire,0,365,0851465F,Identifiant national de l'établissement ; 1re ...


In [16]:
lignes = [
    '# Dictionnaire de données',
    '',
    f'Jeu : `{CLEAN_PATH.name}` · {len(df)} lignes × {df.shape[1]} colonnes',
    '',
    f"**Cible** : `{TARGET}` (régression, taux en %)  ",
    f"**Clé primaire** : `{' × '.join(CLE_PRIMAIRE)}` (unique, 0 conflit)  ",
    "**Source** : INSERSUP, millésime 2026_S1, ministère de l'Enseignement supérieur, "
    'Licence Ouverte Etalab',
    '',
    '| Colonne | Type | Rôle | Manquants | Modalités | Exemple | Description |',
    '|---|---|---|---:|---:|---|---|',
]
for _, ligne in dictionnaire.iterrows():
    valeurs = [f'`{ligne["colonne"]}`', ligne['type'], ligne['rôle'],
               str(ligne['valeurs_manquantes']), str(ligne['modalités']),
               ligne['exemple'], ligne['description']]
    lignes.append('| ' + ' | '.join(str(v).replace('|', '/') for v in valeurs) + ' |')

lignes += [
    '',
    '## Jointure vers une source externe',
    '',
    f"`{CODE_UAI}` est l'identifiant national des établissements : c'est la clé de "
    "jointure vers l'annuaire de l'éducation ou tout référentiel d'établissements.",
    '',
    '## Généré automatiquement',
    '',
    "Ce fichier est produit par `notebooks/etape_3_preparation.ipynb` à partir du jeu "
    'final : il ne peut pas être désynchronisé des données.',
]
DICTIONNAIRE_PATH.write_text('\n'.join(lignes), encoding='utf-8')
print('Dictionnaire écrit :', DICTIONNAIRE_PATH)
print(f'{len(dictionnaire)} colonnes documentées')

Dictionnaire écrit : /home/baptiste/Documents/master LiveCampus/machine_learning/Projet/DATA_DICTIONARY.md
32 colonnes documentées


## 5. Export du jeu final et séparation apprentissage / test

Le jeu nettoyé et enrichi est exporté : c'est le livrable « dataset final » du projet, et
le point d'entrée des étapes 4 et 5.

In [17]:
df.to_csv(CLEAN_PATH, index=False)
print('Jeu final exporté :', CLEAN_PATH.name, df.shape)

X = df[FEATURES].copy()
y = df[TARGET].copy()

# Un seul découpage, un seul random_state, réutilisé à l'identique aux étapes 4 et 5.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

print()
print(f'Apprentissage : {X_train.shape[0]} lignes ({100 * len(X_train) / len(X):.0f} %)')
print(f'Test          : {X_test.shape[0]} lignes ({100 * len(X_test) / len(X):.0f} %)')
print()
controle = pd.DataFrame({
    'apprentissage': y_train.describe().drop('count').round(2),
    'test': y_test.describe().drop('count').round(2),
})
controle['écart'] = (controle['test'] - controle['apprentissage']).round(3)
controle

Jeu final exporté : dataset_phase3_final.csv (17065, 32)

Apprentissage : 13652 lignes (80 %)
Test          : 3413 lignes (20 %)



,apprentissage,test,écart
mean,55.89,55.89,0.00
std,18.45,18.53,0.08
min,0.00,3.12,3.12
25%,43.24,43.10,-0.14
50%,56.10,56.00,-0.10
75%,69.23,69.57,0.34
max,100.00,100.00,0.00


In [18]:
assert TARGET not in FEATURES, 'La cible est dans les variables explicatives'
assert not any('12-' in c or '18-' in c or '24-' in c or '30-' in c for c in FEATURES), \
    'Une variable postérieure à 6 mois est utilisée'
assert not any(c in FEATURES for c in (TX_STABLE, NB_STABLE, CODE_UAI, CODE_SISE)), \
    'Une variable exclue ou un identifiant est utilisé'
assert y.notna().all(), 'La cible contient des valeurs manquantes'
assert X.isna().sum().sum() == 0, 'Les variables explicatives contiennent des manquants'
assert len(X_train) + len(X_test) == len(X), 'Le découpage perd des lignes'
assert set(X_train.index).isdisjoint(X_test.index), 'Apprentissage et test se recouvrent'

print('Tous les contrôles de périmètre et de découpage passent.')
print()
print('Variables transmises à l\'étape 4 :')
for nom in FEATURES:
    marque = ' (créée)' if nom in FEATURES_CREEES else ''
    print(f'  · {nom}{marque}')

Tous les contrôles de périmètre et de découpage passent.

Variables transmises à l'étape 4 :
  · Région
  · Académie
  · Type de diplôme
  · Domaine disciplinaire
  · Discipline
  · Secteur disciplinaire
  · Promotion
  · tranche_effectif (créée)
  · 6-Nombre de sortants - 6 mois après le diplôme
  · 6-Nombre de poursuivants - 6 mois après le diplôme
  · anciennete_promotion (créée)
  · promotion_choc_sanitaire (créée)
  · est_diplome_professionnalisant (créée)
  · taille_formation (créée)
  · ratio_poursuite (créée)
  · taille_mediane_secteur (créée)
  · nb_formations_etablissement (créée)
  · part_sortants_etablissement (créée)
  · nb_etablissements_par_diplome (créée)


### Ce qui n'est volontairement pas fait ici

Aucun encodage, aucune mise à l'échelle, aucune imputation n'est appliquée à ce stade.
Ces trois opérations apprennent quelque chose sur les données : les modalités présentes,
la moyenne et l'écart-type, la valeur de remplacement. Les appliquer avant le découpage
laisserait fuir dans le jeu d'apprentissage une information issue du jeu de test.

Elles sont donc définies à l'étape 4 **à l'intérieur d'un `Pipeline`**, qui les réajuste
sur les seules données d'entraînement de chaque pli de validation croisée.

## Utilisation de l'IA sur cette étape

| Prompt utilisé | Ce que l'IA a produit | Vérification effectuée |
|---|---|---|
| « Quelles features créer à partir d'effectifs de sortants et de poursuivants ? » | Ratio de poursuite, effectif total, tranches | Corrélation de chaque piste mesurée avant de la retenir ; le ratio ressort à −0,31, les tranches beaucoup moins |
| « Est-ce une fuite de données de calculer une moyenne par groupe avant le split ? » | Distinction entre agrégat de `X` et agrégat de `y` | Traduite en test exécutable : permutation de la cible, puis recalcul des agrégats sur le seul train |
| « Comment journaliser un nettoyage de façon traçable ? » | Structure de log JSON par action | Fonction `journaliser` écrite à la main pour forcer une justification à chaque appel |
| « Génère-moi un data dictionary » | Tableau Markdown statique rédigé à la main | Refusé : remplacé par une génération depuis le DataFrame, avec un `assert` qui échoue si une colonne n'est pas décrite |

Le point le plus utile a été le deuxième : la réponse initiale de l'IA affirmait que tout
agrégat calculé avant le découpage est une fuite. C'est faux tant que l'agrégat ne touche
pas la cible, et la nuance a été transformée en deux tests plutôt qu'en affirmation.